<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/06-Evaluate_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating RAG Quality: Hit Rate, MRR, Relevancy & Faithfulness

This notebook walks through a complete evaluation pipeline for a Retrieval-Augmented Generation (RAG) system built on LlamaIndex and ChromaDB.

## What You Will Learn
- How to generate a synthetic QA dataset using the Gemini API
- How to evaluate retrieval quality using Hit Rate and Mean Reciprocal Rank (MRR) via `RetrieverEvaluator`
- How to assess answer quality using Relevancy and Faithfulness metrics via `BatchEvalRunner`
- How to measure factual correctness with `CorrectnessEvaluator`

## Prerequisites
- Familiarity with RAG pipelines (notebooks 02–05)
- LlamaIndex basics (notebook 03)
- ChromaDB as a vector store (notebook 04)
- An OpenAI API key (for embeddings and judge LLM) and a Google API key (for Gemini-based QA generation)


## Install Packages and Setup Variables


In [ ]:
!pip install -q llama-index==0.14.19 openai==2.8.1 google-genai==1.70.0 llama-index-embeddings-openai==0.6.0 llama-index-llms-google-genai==0.9.0 \
                opentelemetry-api==1.38.0 chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [ ]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get('OPENAI_API_KEY', '')
    os.environ["GOOGLE_API_KEY"] = os.environ.get('GOOGLE_API_KEY', '')

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

# Create a Vector Store


In [ ]:
import chromadb

# create client and a new collection
# chromadb.EphemeralClient saves data in-memory.
chroma_client = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = chroma_client.get_or_create_collection("mini-llama-articles")

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore

# Define a storage context object using the created vector database.
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Load the Dataset (CSV)


## Download


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model.


In [ ]:
!curl -o ./mini-dataset.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  169k  100  169k    0     0   456k      0 --:--:-- --:--:-- --:--:--  457k


## Load the Articles


In [ ]:
import csv

rows = []

# Load the CSV file
with open("./mini-dataset.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        rows.append(row)

# The number of characters in the dataset.
len(rows)

14

# Convert to Document obj


In [ ]:
from llama_index.core import Document
from llama_index.core.schema import TextNode

# Convert the chunks to Document objects so the LlamaIndex framework can process them.
documents = [
    Document(
        text=row[1],
        metadata={"title": row[0], "url": row[2], "source_name": row[3]},
    )
    for row in rows
]
# By default, the node/chunks ids are set to random uuids. To ensure same id's per run, we manually set them.
for idx, doc in enumerate(documents):
    doc.id_ = f"doc_{idx}"

In [ ]:
documents[0]

Document(id_='doc_0', embedding=None, metadata={'title': "Beyond GPT-4: What's New?", 'url': 'https://pub.towardsai.net/beyond-gpt-4-whats-new-cbd61a448eb9#dda8', 'source_name': 'towards_ai'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='LLM Variants and Meta\'s Open Source Before shedding light on four major trends, I\'d share the latest Meta\'s Llama 2 and Code Llama. Meta\'s Llama 2 represents a sophisticated evolution in LLMs. This suite spans models pretrained and fine-tuned across a parameter spectrum of 7 billion to 70 billion. A specialized derivative, Llama 2-Chat, has been engineered explicitly for dialogue-centric applications. Benchmarking revealed Llama 2\'s superior performance over most extant open-source chat models. Human-centric evaluations, focusing on safety and utility metrics, positioned Llama 2-Chat as a p

# Transforming


In [ ]:
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.schema import BaseNode
import hashlib


def deterministic_id_func(i: int, doc: BaseNode) -> str:
    """Deterministic ID function for the text splitter.
    This will be used to generate a unique repeatable identifier for each node."""
    unique_identifier = doc.id_ + str(i)
    hasher = hashlib.sha256()
    hasher.update(unique_identifier.encode("utf-8"))
    return hasher.hexdigest()


text_splitter = TokenTextSplitter(
    separator=" ", chunk_size=512, chunk_overlap=128, id_func=deterministic_id_func
)

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        OpenAIEmbedding(model = 'text-embedding-3-small'),
    ],
    vector_store=vector_store,
)

nodes = pipeline.run(documents=documents, show_progress=True)

Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
nodes[0]

TextNode(id_='4ab5bd897f01474fc9b0049f95e31edae3ccd9e74d0f0acd3932b50a74d608b6', embedding=[0.004486083984375, 0.0167083740234375, 0.06158447265625, -0.016204833984375, 0.020477294921875, -0.0224609375, 0.00628662109375, 0.01470947265625, -0.00012314319610595703, 0.005825042724609375, 0.0275115966796875, -0.04571533203125, -0.03533935546875, 0.00426483154296875, -0.035125732421875, -0.0277862548828125, -0.034088134765625, -0.046356201171875, -0.0154266357421875, 0.03765869140625, 0.013092041015625, 0.0072021484375, -0.034515380859375, 0.025909423828125, -0.00504302978515625, -0.02685546875, -0.048065185546875, 0.017547607421875, -0.0175018310546875, -0.0248870849609375, 0.052642822265625, -0.0253753662109375, -0.02215576171875, -0.01171112060546875, -0.0247955322265625, 0.018310546875, -0.005161285400390625, -0.010040283203125, 0.020294189453125, -0.004436492919921875, -0.03228759765625, -0.0626220703125, -0.0020732879638671875, 0.00920867919921875, -0.041015625, 0.00411224365234375, 0

# Load Indexes


In [ ]:
# Create your index
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_vector_store(vector_store)

In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.

from llama_index.llms.google_genai import GoogleGenAI
import google.genai.types as types

config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    max_output_tokens=512,
    temperature=1,
)

llm = GoogleGenAI(model="gemini-2.5-flash",generation_config=config)

query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

In [ ]:
res = query_engine.query("How many parameters LLaMA 2 model has?")

In [ ]:
print(res.response)

The Llama 2 model is available in four sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters. However, the 34 billion parameter model has not yet been released.


In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 9afbdeaea403deb0f61cdc3bca5b4a96afe98f4166b36b4f8606cc41a7c0a4c1
Title	 Fine-Tuning a Llama-2 7B Model for Python Code Generation
Text	 only fine-tuning a small number of additional parameters, with virtually all model parameters remaining frozen. PEFT has been found to produce good generalization with relatively low-volume datasets. Furthermore, it enhances the reusability and portability of the model, as the small checkpoints obtained can be easily added to the base model, and the base model can be easily fine-tuned and reused in multiple scenarios by adding the PEFT parameters. Finally, since the base model is not adjusted, all the knowledge acquired in the pre-training phase is preserved, thus avoiding catastrophic forgetting. Most widely used PEFT techniques aim to keep the pre-trained base model untouched and add new layers or parameters on top of it. These layers are called "Adapters" and the technique of their adjustment "adapter-tuning", we add these layers to the pre

# Evaluate the retrieval process and quality of answers

We can evaluate our RAG system with a dataset of questions and associated chunks. Given a question, we can see if the RAG system retrieves the correct chunks of text that can answer the question.

You can generate a synthetic dataset with an LLM such as `gemini-1.5-flash` or create an authentic and manually curated dataset.

Note that a **well curated dataset will always be a better option**, especially for a specific domain or use case.


In our example, we will generate a synthetic dataset using `gemini-1.5-flash` to make it simple.

This is the default prompt that the `generate_question_context_pairs` function will uses:

```python
DEFAULT_QA_GENERATE_PROMPT_TMPL = """\
Context information is below.

---------------------
{context_str}
---------------------

Given the context information and no prior knowledge,
generate only questions based on the below query.

You are a Teacher/Professor. Your task is to setup \
{num_questions_per_chunk} questions for an upcoming \
quiz/examination. The questions should be diverse in nature \
across the document. Restrict the questions to the \
context information provided."
"""
```


In [ ]:
# Free Tier-Gemini API key
from llama_index.core.llms.utils import LLM
from llama_index.core.schema import MetadataMode, TextNode
from tqdm import tqdm
import json
import re
import uuid
import warnings
import time
from typing import Dict, List, Tuple
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

DEFAULT_QA_GENERATE_PROMPT_TMPL = """\
Context information is below.

---------------------
{context_str}
---------------------

Given the context information and not prior knowledge.
generate only questions based on the below query.

You are a Teacher/ Professor. Your task is to setup \
{num_questions_per_chunk} questions for an upcoming \
quiz/examination. The questions should be diverse in nature \
across the document. Restrict the questions to the \
context information provided."
"""

def generate_question_context_pairs(
    nodes: List[TextNode],
    llm: LLM,
    qa_generate_prompt_tmpl: str = DEFAULT_QA_GENERATE_PROMPT_TMPL,
    num_questions_per_chunk: int = 2,
    request_delay: float = 2.0
) -> EmbeddingQAFinetuneDataset:
    """Generate examples given a set of nodes with delays between requests."""
    node_dict = {
        node.node_id: node.get_content(metadata_mode=MetadataMode.NONE)
        for node in nodes
    }

    queries = {}
    relevant_docs = {}

    for node_id, text in tqdm(node_dict.items()):
        query = qa_generate_prompt_tmpl.format(
            context_str=text, num_questions_per_chunk=num_questions_per_chunk
        )
        response = llm.complete(query)

        result = str(response).strip().split("\n")
        questions = [
            re.sub(r"^\d+[\).\s]", "", question).strip() for question in result
        ]
        questions = [question for question in questions if len(question) > 0][
            :num_questions_per_chunk
        ]

        num_questions_generated = len(questions)
        if num_questions_generated < num_questions_per_chunk:
            warnings.warn(
                f"Fewer questions generated ({num_questions_generated}) "
                f"than requested ({num_questions_per_chunk})."
            )

        for question in questions:
            question_id = str(uuid.uuid4())
            queries[question_id] = question
            relevant_docs[question_id] = [node_id]

        time.sleep(request_delay)

    return EmbeddingQAFinetuneDataset(
        queries=queries, corpus=node_dict, relevant_docs=relevant_docs
    )

#from llama_index.core.evaluation import generate_question_context_pairs

# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.

from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(model="gemini-2.5-flash", temperature=0.3)

rag_eval_dataset = generate_question_context_pairs(
    nodes[:25],
    llm=llm,
    num_questions_per_chunk=1,
    request_delay=4
)

# Save the dataset as a json file for later use
rag_eval_dataset.save_json("./rag_eval_dataset.json")

100%|██████████| 25/25 [03:30<00:00,  8.41s/it]


In [ ]:
# #Paid-Gemini API Key

# from llama_index.core.evaluation import generate_question_context_pairs

# #Define a query engine that is responsible for retrieving related pieces of text,
# #and using a LLM to formulate the final answer.

# from llama_index.llms.google_genai import GoogleGenAI

# import google.genai.types as types

# config = types.GenerateContentConfig(
#     thinking_config=types.ThinkingConfig(thinking_budget=0),
#     max_output_tokens=512,
#     temperature=0.3,
# )

# llm = GoogleGenAI(model="gemini-2.5-flash", generation_config=config)

# rag_eval_dataset = generate_question_context_pairs(nodes, llm=llm, num_questions_per_chunk=1)

# # We can save the dataset as a json file for later use.
# rag_eval_dataset.save_json("./rag_eval_dataset.json")

In [ ]:
# We can also load the dataset from a previously saved json file.
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json("./rag_eval_dataset.json")

### Evaluation for Hit Rate and Mean Reciprocal Rank (MRR)

We will make use of `RetrieverEvaluator` available in Llama-index. We will measure the Hit Rate and Mean Reciprocal Rank (MRR).

**Hit Rate:**

Hit Rate calculates the fraction of queries where the correct answer is found within the top-k retrieved documents. In simpler terms, it's about how often our system gets it right within the top few guesses.

**Mean Reciprocal Rank (MRR):**

For each query, MRR evaluates the system's accuracy by looking at the rank of the highest-placed relevant document. Specifically, it's the average of the reciprocal ranks of the first relevant document for a set of queries. If the first relevant document is the top result, the reciprocal rank is 1; if it's second, the reciprocal rank is 1/2; and so on.
In summary, Hit Rate tells you how often the system gets it right in its top guesses, and MRR tells you how close to the top the right answer usually is. Both metrics are useful for evaluating the effectiveness of a retrieval system, like how well a search engine or a recommendation system works.


In [ ]:
import pandas as pd


def display_results_retriever(name, eval_results):
    """Display results from evaluate."""

    metric_dicts = []
    for eval_result in eval_results:
        metric_dict = eval_result.metric_vals_dict
        metric_dicts.append(metric_dict)

    full_df = pd.DataFrame(metric_dicts)

    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()

    metric_df = pd.DataFrame(
        {"Retriever Name": [name], "Hit Rate": [hit_rate], "MRR": [mrr]}
    )

    return metric_df

In [ ]:
from llama_index.core.evaluation import RetrieverEvaluator

# We can evaluate the retrievers with different top_k values.
for i in [2, 4, 6, 8, 10]:
    retriever = index.as_retriever(similarity_top_k=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(
        rag_eval_dataset, workers=32
    )
    print(display_results_retriever(f"Retriever top_{i}", eval_results))

time.sleep(60)

    Retriever Name  Hit Rate  MRR
0  Retriever top_2       0.8  0.7
    Retriever Name  Hit Rate       MRR
0  Retriever top_4      0.92  0.733333
    Retriever Name  Hit Rate       MRR
0  Retriever top_6      0.92  0.733333
    Retriever Name  Hit Rate       MRR
0  Retriever top_8      0.92  0.733333
     Retriever Name  Hit Rate       MRR
0  Retriever top_10      0.96  0.737333


### Evaluation using Relevance and Faithfulness metrics.

Here, we evaluate the answer generated by the LLM. Is the answer using the correct context? Is the answer faithful to the context? Is the answer relevant to the question?

An LLM will answer these questions, more specifically `gpt-5`.

**`FaithfulnessEvaluator`**
Evaluates if the answer is faithful to the retrieved contexts (in other words, whether there's an hallucination).

**`RelevancyEvaluator`**
Evaluates whether the retrieved context and answer are relevant to the user question.

Now, let's see how the top_k value affects these two metrics.


In [ ]:
from llama_index.core.evaluation import RelevancyEvaluator, FaithfulnessEvaluator, BatchEvalRunner
from llama_index.llms.openai import OpenAI

# Create your index
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex.from_vector_store(vector_store)

# Define an LLM as a judge
llm_gpt5 = OpenAI(model="gpt-5.4-mini", additional_kwargs={"reasoning_effort": "low"})
llm_gpt5_mini = OpenAI(model="gpt-5.4-mini", additional_kwargs={"reasoning_effort": "low"})

# Initiate the faithfulness and relevancy evaluator objects
faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_gpt5)
relevancy_evaluator = RelevancyEvaluator(llm=llm_gpt5)

# Extract the questions from the dataset
queries = list(rag_eval_dataset.queries.values())
# Limit to first 10 question to save time (!!remove this line in production!!)
batch_eval_queries = queries[:20]

# The batch evaluator runs the evaluation in batches
runner = BatchEvalRunner(
    {"faithfulness": faithfulness_evaluator, "relevancy": relevancy_evaluator},
    workers=32,
)


# Define a for-loop to try different `similarity_top_k` values
for i in [2, 4, 6, 8, 10]:
    # Set query engine with different number of returned chunks
    query_engine = index.as_query_engine(similarity_top_k=i, llm = llm_gpt5_mini)

    # Run the evaluation
    eval_results = await runner.aevaluate_queries(query_engine, queries=batch_eval_queries)

    # Printing the results
    faithfulness_score = sum(
        result.passing for result in eval_results["faithfulness"]
    ) / len(eval_results["faithfulness"])
    print(f"top_{i} faithfulness_score: {faithfulness_score}")

    relevancy_score = sum(result.passing for result in eval_results["relevancy"]) / len(
        eval_results["relevancy"]
    )
    print(f"top_{i} relevancy_score: {relevancy_score}")
    print("="*15)

top_2 faithfulness_score: 0.9
top_2 relevancy_score: 1.0
top_4 faithfulness_score: 0.9
top_4 relevancy_score: 1.0
top_6 faithfulness_score: 0.85
top_6 relevancy_score: 1.0
top_8 faithfulness_score: 0.8
top_8 relevancy_score: 1.0
top_10 faithfulness_score: 0.9
top_10 relevancy_score: 0.95


## Optional: Evaluation Metrics Summary

In [ ]:
# Summarize all evaluation metrics into a single comparison table
import pandas as pd

summary = []

# Add retriever metrics if available
if 'retriever_eval_results' in dir() or 'eval_results' in locals():
    try:
        hit_rates = [r.is_hit for r in eval_results if hasattr(r, 'is_hit')]
        if hit_rates:
            summary.append({"Metric": "Hit Rate", "Score": f"{sum(hit_rates)/len(hit_rates):.3f}", "Description": "Fraction of queries with a relevant doc in top-k"})
    except Exception:
        pass

# Add answer quality metrics if available
metrics_to_check = [
    ("relevancy_results", "Relevancy", "Answer is relevant to the question"),
    ("faithfulness_results", "Faithfulness", "Answer is grounded in retrieved context"),
]
for var_name, metric_name, desc in metrics_to_check:
    results = locals().get(var_name) or globals().get(var_name)
    if results:
        try:
            scores = [float(r.score) for r in results if hasattr(r, 'score') and r.score is not None]
            if scores:
                summary.append({"Metric": metric_name, "Score": f"{sum(scores)/len(scores):.3f}", "Description": desc})
        except Exception:
            pass

if summary:
    df_summary = pd.DataFrame(summary)
    print("=== RAG Evaluation Summary ===")
    print(df_summary.to_string(index=False))
else:
    print("Run the evaluation cells above to populate the summary.")

Run the evaluation cells above to populate the summary.


### Correctness


In [ ]:
from llama_index.core.evaluation import CorrectnessEvaluator

query = (
    "Can you explain the theory of relativity proposed by Albert Einstein in" " detail?"
)

reference = """
Certainly! Albert Einstein's theory of relativity consists of two main components: special relativity and general relativity. Special relativity, published in 1905, introduced the concept that the laws of physics are the same for all non-accelerating observers and that the speed of light in a vacuum is a constant, regardless of the motion of the source or observer. It also gave rise to the famous equation E=mc², which relates energy (E) and mass (m).

General relativity, published in 1915, extended this to include gravity. Einstein proposed that massive objects cause a distortion in spacetime, which is felt as gravity. This was a revolutionary departure from Newton's law of gravitation. The rubber-sheet analogy is often used: imagine placing a heavy ball on a stretched rubber sheet. The ball causes the sheet to curve, and if you then roll a smaller ball nearby, it will spiral inward toward the heavy ball, mimicking the effect of gravity.

In essence, general relativity provided a new understanding of gravity, explaining phenomena like the bending of light by gravity (gravitational lensing) and the precession of the orbit of Mercury. It has been confirmed through numerous experiments and observations and has become a fundamental theory in modern physics.
"""

response = """
Certainly! Albert Einstein's theory of relativity consists of two main components: special relativity and general relativity. Special relativity, published in 1905, introduced the concept that the laws of physics are the same for all non-accelerating observers and that the speed of light in a vacuum is a constant, regardless of the motion of the source or observer. It also gave rise to the famous equation E=mc², which relates energy (E) and mass (m).

General relativity, published in 1915, extended this to include gravity. Einstein proposed that massive objects generate powerful magnetic fields that warp spacetime, which is felt as gravity. The rubber-sheet with magnets analogy is often used: imagine placing a magnet on a stretched rubber sheet. The magnetic field causes the sheet to curve, and if you then roll a smaller ball nearby, it will spiral inward, mimicking the effect of gravity.

In essence, general relativity provided a new understanding of gravity. It has been confirmed through numerous experiments and observations and has become a fundamental theory in modern physics.
"""

In [ ]:
evaluator = CorrectnessEvaluator(llm=llm_gpt5)

result = evaluator.evaluate(query=query,response=response,reference=reference,)

In [ ]:
print("Score:", result.score)

Score: 3.0


In [ ]:
print("Feedback:", result.feedback)

Feedback: The answer is relevant and captures the basics of special and general relativity, but it includes a significant scientific error by saying massive objects generate magnetic fields to warp spacetime. It also uses an incorrect analogy.


## Optional: Per-Query Score Breakdown

In [ ]:
# Show which individual queries scored lowest — useful for debugging retrieval gaps
print("=== Per-Query Correctness Scores ===\n")

# Use the correctness evaluator result from the cell above
try:
    print(f"Query   : {query}")
    print(f"Score   : {result.score}/2.0")
    print(f"Feedback: {result.feedback}")
    print()
    if float(result.score) >= 1.5:
        print("Assessment: Good — answer is correct and well-grounded.")
    elif float(result.score) >= 1.0:
        print("Assessment: Partial — answer is on topic but missing details.")
    else:
        print("Assessment: Poor — consider improving chunking or retrieval top_k.")
except Exception as e:
    print(f"Run the CorrectnessEvaluator cell first. ({e})")

=== Per-Query Correctness Scores ===

Query   : Can you explain the theory of relativity proposed by Albert Einstein in detail?
Score   : 3.0/2.0
Feedback: The answer is relevant and captures the basics of special and general relativity, but it includes a significant scientific error by saying massive objects generate magnetic fields to warp spacetime. It also uses an incorrect analogy.

Assessment: Good — answer is correct and well-grounded.
